In [65]:
import pandas as pd
import numpy as np
import os

In [66]:
# load already processed hud data (2014–2022)
hud_df = pd.read_csv("data/clean_hcv_data.csv", dtype={"code": str})
hud_df.head()

,program_label,program,entities,code,number_reported,rent_per_month,hh_income,tpoverty,treatment,year,post,poverty_indicator
0,Housing Choice Vouchers,3,TX Texas 113 Dallas County 48113000401,48113000401,39,297,11131,38,1,2014,0,1
1,Housing Choice Vouchers,3,TX Texas 113 Dallas County 48113000405,48113000405,17,268,10446,40,1,2014,0,1
2,Housing Choice Vouchers,3,TX Texas 113 Dallas County 48113000500,48113000500,25,202,8242,17,1,2014,0,0
3,Housing Choice Vouchers,3,TX Texas 113 Dallas County 48113000601,48113000601,17,336,12377,35,1,2014,0,1
4,Housing Choice Vouchers,3,TX Texas 113 Dallas County 48113000800,48113000800,123,326,11988,27,1,2014,0,0


In [67]:
# Define tract filtering rules for treatment and control groups
treatment_prefixes = ["48113"]
control_prefixes = ["48201"]

# Columns required for analysis
columns_to_keep = [
    "program_label",
    "program",
    "entities",
    "code",
    "number_reported",
    "rent_per_month",
    "hh_income",
    "tpoverty",
]

# Columns that require numeric conversion
numeric_cols = ["number_reported", "rent_per_month", "hh_income", "tpoverty"]

# Known missing value codes in raw data
missing_codes = ["-1", "-4", "-5", "NA"]

# HUD suppression threshold for small counts
suppression_threshold = 11

In [68]:
def filter_tracts(df, treatment_prefixes, control_prefixes, file_name="unknown"):
    """
    Filter dataset to include only treatment and control tracts and create treatment indicator.

    Parameters:
        df (pd.DataFrame): input dataset
        treatment_prefixes (list): tract prefixes for treatment group
        control_prefixes (list): tract prefixes for control group
        file_name (str): file identifier for debugging

    Returns:
        pd.DataFrame: filtered dataset with treatment indicator
    """
    try:
        df["code"] = df["code"].astype(str)

        # Create treatment indicator
        df["treatment"] = (
            df["code"].str.startswith(tuple(treatment_prefixes)).astype(int)
        )

        # Keep only relevant tracts
        valid_prefixes = tuple(treatment_prefixes + control_prefixes)
        df = df[df["code"].str.startswith(valid_prefixes)]

        return df

    except Exception as e:
        print(f"Error in filter_tracts for file: {file_name}")
        print(e)
        return df


def processing_missing_values(df, file_name="unknown"):
    """
    Clean dataset by handling missing values, suppression rules, and numeric conversion.

    Parameters:
        df (pd.DataFrame): input dataset
        file_name (str): file identifier for debugging

    Returns:
        pd.DataFrame: cleaned dataset
    """
    try:
        df["code"] = df["code"].astype(str)

        # Remove invalid tract codes
        df.loc[df["code"].str.endswith("999999"), "code"] = np.nan

        # Convert outcome variable
        df["number_reported"] = pd.to_numeric(df["number_reported"], errors="coerce")

        # Apply suppression rule
        df.loc[df["number_reported"] < suppression_threshold, "number_reported"] = np.nan

        # Replace known missing codes
        df = df.replace(missing_codes, np.nan)

        # Convert numeric columns
        for col in numeric_cols:
            if col in df.columns:
                df[col] = pd.to_numeric(df[col], errors="coerce")

        # Drop rows with missing values
        df = df.dropna()

        return df

    except Exception as e:
        print(f"Error in processing_missing_values for file: {file_name}")
        print(e)
        return df


def add_year(df, year):
    """
    Add a year column to the dataset.

    Parameters:
        df (pd.DataFrame): input dataset
        year (int): year value to assign

    Returns:
        pd.DataFrame: dataset with year column
    """
    df["year"] = year
    return df


def create_variables(df):
    """
    Create derived variables for analysis.

    Parameters:
        df (pd.DataFrame): input dataset

    Returns:
        pd.DataFrame: dataset with derived variables
    """
    # Post indicator for policy period
    df["post"] = (df["year"] >= 2018).astype(int)

    # Poverty indicator based on 30 percent threshold
    df["poverty_indicator"] = (df["tpoverty"] >= 30).astype(int)

    return df

In [69]:
# Load raw 2013 hud file
file_path = "data/TRACT_MO_WY_2013.xlsx"

df = pd.read_excel(file_path, dtype=str)

df.head()

,gsl,states,entities,sumlevel,program_label,program,sub_program,NAME,code,total_units,...,tminority,tpct_ownsfd,fedhse,CBSA,PLACE,latitude,longitude,STATE,pha_total_units,ha_size
0,Census Tract,MO Missouri,MO Adair County Census Tract 9501 29001950100,7,Public Housing,2,A,Census Tract 9501,29001950100,NaN,...,-1,-1,A,28860,08002,-1,-1,MO,-1,A
1,Census Tract,MO Missouri,MO Adair County Census Tract 9502 29001950200,7,Public Housing,2,A,Census Tract 9502,29001950200,NaN,...,-1,-1,A,28860,53534,-1,-1,MO,-1,A
2,Census Tract,MO Missouri,MO Adair County Census Tract 9503 29001950300,7,Public Housing,2,A,Census Tract 9503,29001950300,NaN,...,-1,-1,A,28860,39026,-1,-1,MO,-1,A
3,Census Tract,MO Missouri,MO Adair County Census Tract 9504 29001950400,7,Public Housing,2,A,Census Tract 9504,29001950400,NaN,...,-1,-1,A,28860,39026,-1,-1,MO,-1,A
4,Census Tract,MO Missouri,MO Adair County Census Tract 9505 29001950500,7,Public Housing,2,A,Census Tract 9505,29001950500,93,...,8,62,A,28860,39026,-1,-1,MO,-1,A


In [70]:
# Keep only housing choice voucher program observations
df = df[df["program"] == "3"]

# Retain relevant variables for analysis
df = df[[col for col in columns_to_keep if col in df.columns]]

df.head()

,program_label,program,entities,code,number_reported,rent_per_month,hh_income,tpoverty
1394,Housing Choice Vouchers,3,MO Adair County Census Tract 9501 29001950100,29001950100,11,229.15347442093,10124.6565572405,16
1395,Housing Choice Vouchers,3,MO Adair County Census Tract 9502 29001950200,29001950200,16,198.5,8108.1875,15
1396,Housing Choice Vouchers,3,MO Adair County Census Tract 9503 29001950300,29001950300,49,264.503563746784,10173.8776938973,43
1397,Housing Choice Vouchers,3,MO Adair County Census Tract 9504 29001950400,29001950400,41,274.235928564262,11544.5139961874,14
1398,Housing Choice Vouchers,3,MO Adair County Census Tract 9505 29001950500,29001950500,74,278.680964699991,10958.9318757128,27


In [71]:
# Apply full preprocessing on 2013 HCV data

df = filter_tracts(df, treatment_prefixes, control_prefixes, file_path)

df = processing_missing_values(df, file_path)

df = add_year(df, 2013)

df = create_variables(df)

df.head()

,program_label,program,entities,code,number_reported,rent_per_month,hh_income,tpoverty,treatment,year,post,poverty_indicator
160822,Housing Choice Vouchers,3,TX Dallas County Census Tract 100 48113010000,48113010000,148.0,329.739039,12917.223655,23.0,1,2013,0,0
160823,Housing Choice Vouchers,3,TX Dallas County Census Tract 101.01 48113010101,48113010101,71.0,380.181672,13468.052117,39.0,1,2013,0,1
160828,Housing Choice Vouchers,3,TX Dallas County Census Tract 107.01 48113010701,48113010701,56.0,386.895806,12941.244816,24.0,1,2013,0,0
160829,Housing Choice Vouchers,3,TX Dallas County Census Tract 107.03 48113010703,48113010703,116.0,269.853153,9610.718526,28.0,1,2013,0,0
160830,Housing Choice Vouchers,3,TX Dallas County Census Tract 107.04 48113010704,48113010704,61.0,331.821016,12901.027552,32.0,1,2013,0,1


In [72]:
# Combine 2013 data with previously processed hud panel
hud_full = pd.concat([df, hud_df], ignore_index=True)

# Drop unused financial variables
hud_full = hud_full.drop(columns=["rent_per_month", "hh_income"], errors="ignore")

hud_full.head()

,program_label,program,entities,code,number_reported,tpoverty,treatment,year,post,poverty_indicator
0,Housing Choice Vouchers,3,TX Dallas County Census Tract 100 48113010000,48113010000,148.0,23.0,1,2013,0,0
1,Housing Choice Vouchers,3,TX Dallas County Census Tract 101.01 48113010101,48113010101,71.0,39.0,1,2013,0,1
2,Housing Choice Vouchers,3,TX Dallas County Census Tract 107.01 48113010701,48113010701,56.0,24.0,1,2013,0,0
3,Housing Choice Vouchers,3,TX Dallas County Census Tract 107.03 48113010703,48113010703,116.0,28.0,1,2013,0,0
4,Housing Choice Vouchers,3,TX Dallas County Census Tract 107.04 48113010704,48113010704,61.0,32.0,1,2013,0,1


In [73]:
def assign_period(year):
    """
    Assign a five-year period label based on year.

    Parameters:
        year (int): observation year

    Returns:
        str: period label
    """
    if 2013 <= year <= 2017:
        return "2013-2017"
    elif 2018 <= year <= 2022:
        return "2018-2022"
    return np.nan


# Create period variable
hud_full["year"] = hud_full["year"].astype(int)
hud_full["period"] = hud_full["year"].apply(assign_period)

# Remove observations outside target periods
hud_full = hud_full.dropna(subset=["period"])

# Aggregate to tract-period level
hud_agg = (
    hud_full
    .groupby(["code", "period"], as_index=False)
    .agg({
        "number_reported": "mean",
        "tpoverty": "mean"
    })
)

# Recreate poverty indicator from aggregated values
hud_agg["poverty_indicator"] = (hud_agg["tpoverty"] >= 30).astype(int)

# Rename period for merge consistency
hud_agg = hud_agg.rename(columns={"period": "year"})

hud_agg.head()

,code,year,number_reported,tpoverty,poverty_indicator
0,48113000401,2013-2017,35.00,35.0,1
1,48113000401,2018-2022,35.20,32.0,1
2,48113000405,2013-2017,17.60,36.8,1
3,48113000405,2018-2022,18.50,26.5,0
4,48113000406,2018-2022,14.75,18.5,0


In [74]:
# Load racial composition datasets for both time periods
rc_1317 = pd.read_csv("data/nhgis0001_ds233_20175_tract.csv", dtype=str)
rc_1822 = pd.read_csv("data/nhgis0003_ds262_20225_tract.csv", dtype=str)

In [75]:
def process_rc(df, rename_dict, year_label):
    """
    Clean and standardize nhgis racial composition data.

    Steps:
        - Select relevant variables
        - Rename race categories
        - Create tract identifier from geoid
        - Filter to treatment and control tracts
        - Convert numeric fields
        - Drop selected race categories
        - Assign period label

    Parameters:
        df (pd.DataFrame): raw nhgis dataset
        rename_dict (dict): column renaming dictionary
        year_label (str): period label

    Returns:
        pd.DataFrame: cleaned racial composition dataset
    """
    # Select required columns
    cols = ["YEAR", "GEOID"] + list(rename_dict.keys())
    df = df[cols]

    # Rename race variables
    df = df.rename(columns=rename_dict)

    # Ensure geoid is string before processing
    df["GEOID"] = df["GEOID"].astype(str).str.strip()

    # Extract tract code (11-digit census tract id)
    df["code"] = df["GEOID"].str.extract(r"(\d{11})")

    # Filter to treatment and control tracts
    valid_prefixes = ("48113", "48201")
    df = df[df["code"].str.startswith(valid_prefixes)]

    # Convert race columns to numeric
    race_cols = list(rename_dict.values())
    for col in race_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    # Drop selected race categories
    df = df.drop(
        columns=[
            "Native_Hawaiian_Pacific_Islander_alone",
            "American_Indian_Alaska_Native_alone"
        ],
        errors="ignore"
    )

    # Assign period label
    df["year"] = year_label

    return df

In [76]:
rename_1317 = {
    "AHY2E001": "Total",
    "AHY2E002": "White_alone",
    "AHY2E003": "Black_or_African_American_alone",
    "AHY2E004": "American_Indian_Alaska_Native_alone",
    "AHY2E005": "Asian_alone",
    "AHY2E006": "Native_Hawaiian_Pacific_Islander_alone",
    "AHY2E007": "Some_other_race_alone",
    "AHY2E008": "Two_or_more_races",
    "AHY2E009": "Two_races_including_some_other_race",
    "AHY2E010": "Two_races_excluding_some_other_and_three_plus"
}

rename_1822 = {
    "AQNGE001": "Total",
    "AQNGE002": "White_alone",
    "AQNGE003": "Black_or_African_American_alone",
    "AQNGE004": "American_Indian_Alaska_Native_alone",
    "AQNGE005": "Asian_alone",
    "AQNGE006": "Native_Hawaiian_Pacific_Islander_alone",
    "AQNGE007": "Some_other_race_alone",
    "AQNGE008": "Two_or_more_races",
    "AQNGE009": "Two_races_including_some_other_race",
    "AQNGE010": "Two_races_excluding_some_other_and_three_plus"
}

In [77]:
# Rename GEO_ID to GEOID for consistency
rc_1822 = rc_1822.rename(columns={"GEO_ID": "GEOID"})

# Process racial composition data
rc_1317_clean = process_rc(rc_1317, rename_1317, "2013-2017")
rc_1822_clean = process_rc(rc_1822, rename_1822, "2018-2022")

In [78]:
# Combine racial composition data into a single dataset

rc_full = pd.concat([rc_1317_clean, rc_1822_clean], ignore_index=True)

rc_full.head()

,YEAR,GEOID,Total,White_alone,Black_or_African_American_alone,Asian_alone,Some_other_race_alone,Two_or_more_races,Two_races_including_some_other_race,Two_races_excluding_some_other_and_three_plus,code,year
0,2013-2017,14000US48113000100,4105,3481,304,128,33,118,20,98,48113000100,2013-2017
1,2013-2017,14000US48113000201,2949,2820,15,0,80,34,0,34,48113000201,2013-2017
2,2013-2017,14000US48113000202,3857,3491,49,101,92,90,0,90,48113000202,2013-2017
3,2013-2017,14000US48113000300,4235,3673,124,221,44,145,40,105,48113000300,2013-2017
4,2013-2017,14000US48113000401,5755,3426,805,758,636,119,24,95,48113000401,2013-2017


In [79]:
# Combine aggregated HCV data with racial composition data 

final_df = pd.merge(
    hud_agg,
    rc_full,
    on=["code", "year"],
    how="inner"
)

# Drop unused columns from merged dataset
final_df = final_df.drop(columns=["YEAR", "GEOID"], errors="ignore")

final_df.head(575)

,code,year,number_reported,tpoverty,poverty_indicator,Total,White_alone,Black_or_African_American_alone,Asian_alone,Some_other_race_alone,Two_or_more_races,Two_races_including_some_other_race,Two_races_excluding_some_other_and_three_plus
0,48113000401,2013-2017,35.0,35.0,1,5755,3426,805,758,636,119,24,95
1,48113000401,2018-2022,35.2,32.0,1,4649,2253,1110,465,356,367,322,45
2,48113000405,2013-2017,17.6,36.8,1,2282,1315,474,279,173,41,0,41
3,48113000405,2018-2022,18.5,26.5,0,1971,845,864,79,21,162,95,67
4,48113000500,2013-2017,23.5,20.5,0,5357,3901,475,377,349,235,0,235
...,...,...,...,...,...,...,...,...,...,...,...,...,...
570,48201220600,2013-2017,17.6,28.4,0,3784,2865,527,0,299,63,35,28
571,48201220600,2018-2022,12.0,29.0,0,3137,998,672,28,710,729,729,0
572,48201220700,2013-2017,104.8,39.4,1,6128,4252,1489,0,378,9,9,0
573,48201220800,2013-2017,47.8,53.0,1,3368,1877,1353,0,115,9,0,9


In [80]:
final_df.shape

(1146, 13)

In [81]:
# Save final merged dataset
final_df.to_csv("data/clean_rc_data.csv", index=False)